In [1]:
# ── Cell 1: Mount Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


The below code basically just, try to sample from the whole distribution, hence fails for class specific generation

In [2]:
# ============================================================
# scVI — Generate Synthetic Cells
# Google Colab | TensorFlow 2
# ============================================================

!pip install -q scikit-learn

# ── Cell 2: Imports ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print(f'TensorFlow: {tf.__version__}')

# ── Cell 3: Configuration — EDIT PATHS ───────────────────────
DATASET = 'PDO'   # 'PBMC' or 'PDO'

REAL_DATA = {
    'PBMC': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv',
    },
    'PDO': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Stem_High_Raw_Finalized.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Differential_Low_Raw_Finalized.csv',
    },
}

# Checkpoint dir — must contain best_model.index + best_model.data-00000-of-00001
CKPT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scVI Approach/PBMC/checkpoints',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scVI Approach/PDO/checkpoints',
}

OUTPUT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scVI/PBMC',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/scVI/PDO',
}

N_GENERATE  = 10000
MIN_PROB    = 0.90
N_PER_CLASS = {'PBMC': 1186, 'PDO': 1415}

CLASS_NAMES = {
    'PBMC': {0: 'b',         1: 'mono'},
    'PDO' : {0: 'stem_high', 1: 'diff_low'},
}

CFG  = dict(n_latent=10, n_hidden=128, n_layers=1, dropout_rate=0.1, dispersion='gene')
EPS  = 1e-8
SEED = 42

# ── Cell 4: scVI Architecture ─────────────────────────────────

class FCBlock(keras.layers.Layer):
    def __init__(self, n_out, dropout_rate=0.1, **kw):
        super().__init__(**kw)
        self.dense = keras.layers.Dense(n_out)
        self.bn    = keras.layers.BatchNormalization(momentum=0.99, epsilon=0.001)
        self.relu  = keras.layers.ReLU()
        self.drop  = keras.layers.Dropout(dropout_rate)
    def call(self, x, training=False):
        return self.drop(self.relu(self.bn(self.dense(x), training=training)),
                         training=training)

class scVI(keras.Model):
    def __init__(self, n_input, n_hidden=128, n_latent=10, n_layers=1,
                 dropout_rate=0.1, **kw):
        super().__init__(**kw)
        self.n_latent  = n_latent
        self.n_input   = n_input
        self.log_theta = tf.Variable(tf.random.normal([n_input]),
                                     trainable=True, name='log_theta')
        self.kl_weight = tf.Variable(0.0, trainable=False)

        self.enc_layers = [FCBlock(n_hidden, dropout_rate) for _ in range(n_layers)]
        self.l_mean_fc  = keras.layers.Dense(1)
        self.l_var_fc   = keras.layers.Dense(1, activation='softplus')
        self.z_mean_fc  = keras.layers.Dense(n_latent)
        self.z_var_fc   = keras.layers.Dense(n_latent, activation='softplus')

        self.dec_layers    = [FCBlock(n_hidden, dropout_rate) for _ in range(n_layers)]
        self.px_scale_fc   = keras.layers.Dense(n_input, activation='softmax')
        self.px_dropout_fc = keras.layers.Dense(n_input)

    def encode(self, x, training=False):
        h = tf.math.log1p(x)
        for layer in self.enc_layers:
            h = layer(h, training=training)
        return (self.z_mean_fc(h), self.z_var_fc(h) + EPS,
                self.l_mean_fc(h), self.l_var_fc(h) + EPS)

    def decode(self, z, log_l, training=False):
        h = z
        for layer in self.dec_layers:
            h = layer(h, training=training)
        px_rate = tf.exp(log_l) * self.px_scale_fc(h)
        return px_rate, self.px_dropout_fc(h)

    def reparameterise(self, mean, var):
        return mean + tf.sqrt(var) * tf.random.normal(tf.shape(mean))

    def compute_elbo(self, x, l_mean_prior, l_var_prior, training=False):
        z_mean, z_var, l_mean, l_var = self.encode(x, training=training)
        z     = self.reparameterise(z_mean, z_var)
        log_l = self.reparameterise(l_mean, l_var)
        px_rate, px_dropout = self.decode(z, log_l, training=training)
        theta = tf.exp(self.log_theta)
        recon = self._log_zinb(x, px_rate, theta, px_dropout)
        kl_z  = -0.5 * tf.reduce_sum(
            1 + tf.math.log(z_var) - tf.square(z_mean) - z_var, axis=1)
        kl_l  = -0.5 * tf.reduce_sum(
            1 + tf.math.log(z_var + EPS) - tf.math.log(l_var_prior + EPS)
            - (tf.square(l_mean - l_mean_prior) + z_var) / (l_var_prior + EPS), axis=1)
        elbo  = -tf.reduce_mean(recon) + self.kl_weight * tf.reduce_mean(kl_z + kl_l)
        return elbo, recon, kl_z, kl_l

    def _log_zinb(self, x, mu, theta, pi_logits):
        softplus_pi      = tf.nn.softplus(-pi_logits)
        log_theta_mu_eps = tf.math.log(theta + mu + EPS)
        nb_term = (theta * (tf.math.log(theta + EPS) - log_theta_mu_eps)
                   + x * (tf.math.log(mu + EPS) - log_theta_mu_eps)
                   + tf.math.lgamma(x + theta)
                   - tf.math.lgamma(theta)
                   - tf.math.lgamma(x + 1))
        log_p = tf.where(x < EPS,
                         tf.math.softplus(nb_term - pi_logits + softplus_pi) - softplus_pi,
                         nb_term - pi_logits + softplus_pi)
        return tf.reduce_sum(log_p, axis=1)

    def generate(self, n_cells, l_mean_prior, l_std_prior):
        z     = tf.random.normal([n_cells, self.n_latent])
        log_l = tf.random.normal([n_cells, 1]) * l_std_prior + l_mean_prior
        px_rate, _ = self.decode(z, log_l, training=False)
        theta = tf.exp(self.log_theta)
        concentration = tf.broadcast_to(theta, tf.shape(px_rate))
        rate          = theta / (px_rate + EPS)
        gamma_samples = tf.random.gamma(shape=[], alpha=concentration, beta=rate)
        counts        = tf.random.poisson(shape=[], lam=gamma_samples + EPS)
        return counts.numpy().astype(np.float32)

# ── Cell 5: Load Real Data ────────────────────────────────────

print(f'Loading real {DATASET} data...')
Xs, ys = [], []
for label_idx, path in REAL_DATA[DATASET].items():
    df = pd.read_csv(path, header=None)
    Xs.append(df.values.astype(np.float32))
    ys.append(np.full(len(df), label_idx, dtype=np.int32))
    print(f'  Class {label_idx}: {df.shape}')

X_real  = np.concatenate(Xs)
y_real  = np.concatenate(ys)
N_GENES = X_real.shape[1]
log_lib  = np.log(X_real.sum(axis=1) + EPS)
lib_mean = float(log_lib.mean())
lib_std  = float(log_lib.std())
print(f'Combined: {X_real.shape} | lib_mean={lib_mean:.3f}, lib_std={lib_std:.3f}')

# ── Cell 6: Build scVI & Restore Checkpoint ──────────────────

tf.random.set_seed(SEED)
model = scVI(N_GENES, CFG['n_hidden'], CFG['n_latent'],
             CFG['n_layers'], CFG['dropout_rate'], name=f'scVI_{DATASET}')

# Build weights via dummy forward pass
_ = model.compute_elbo(tf.zeros([2, N_GENES]), tf.zeros([2,1]), tf.ones([2,1]))
print('scVI model ready.')

ckpt   = tf.train.Checkpoint(model=model)
status = ckpt.restore(os.path.join(CKPT_DIR[DATASET], 'best_model')).expect_partial()
print('Checkpoint restored.')

# ── Cell 7: Generate Pool ─────────────────────────────────────

print(f'\nGenerating {N_GENERATE} synthetic cells...')
X_synth = model.generate(N_GENERATE, lib_mean, lib_std)
print(f'  Shape   : {X_synth.shape}')
print(f'  Sparsity: {(X_synth == 0).mean()*100:.1f}%')

# ── Cell 8: RF Classifier on Real Data ───────────────────────

print('\nTraining RF classifier on real data only...')
X_real_log  = np.log1p(X_real)
X_synth_log = np.log1p(X_synth)

rf = RandomForestClassifier(n_estimators=200, max_features='sqrt',
                             n_jobs=-1, random_state=SEED)
rf.fit(X_real_log, y_real)
print(classification_report(y_real, rf.predict(X_real_log),
      target_names=list(CLASS_NAMES[DATASET].values())))

# ── Cell 9: Classify, Filter, Sample, Save ───────────────────

probs    = rf.predict_proba(X_synth_log)
labels   = rf.predict(X_synth_log)
max_prob = probs.max(axis=1)

print(f'High-confidence (>={MIN_PROB}): {(max_prob >= MIN_PROB).sum()} / {N_GENERATE}')

os.makedirs(OUTPUT_DIR[DATASET], exist_ok=True)
rng = np.random.default_rng(SEED)

for class_idx, class_name in CLASS_NAMES[DATASET].items():
    mask       = (labels == class_idx) & (max_prob >= MIN_PROB)
    candidates = X_synth[mask]
    n          = N_PER_CLASS[DATASET]

    print(f'\nClass {class_idx} ({class_name}): {len(candidates)} candidates')

    if len(candidates) < n:
        print(f'  WARNING: only {len(candidates)} available, need {n}. Lower MIN_PROB.')
        selected = candidates
    else:
        selected = candidates[rng.choice(len(candidates), size=n, replace=False)]

    print(f'  Selected : {selected.shape}')
    print(f'  Sparsity : {(selected == 0).mean()*100:.1f}%')

    out_path = os.path.join(OUTPUT_DIR[DATASET],
                            f'scvi_synthetic_{class_name}_expression.csv')
    pd.DataFrame(selected).to_csv(out_path, header=False, index=False)
    print(f'  Saved → {out_path}')

print('\nDone.')

TensorFlow: 2.20.0
Loading real PDO data...
  Class 0: (1415, 1600)
  Class 1: (1415, 1600)
Combined: (2830, 1600) | lib_mean=3.470, lib_std=1.380
scVI model ready.
Checkpoint restored.

Generating 10000 synthetic cells...
  Shape   : (10000, 1600)
  Sparsity: 95.8%

Training RF classifier on real data only...
              precision    recall  f1-score   support

   stem_high       1.00      1.00      1.00      1415
    diff_low       1.00      1.00      1.00      1415

    accuracy                           1.00      2830
   macro avg       1.00      1.00      1.00      2830
weighted avg       1.00      1.00      1.00      2830

High-confidence (>=0.9): 5383 / 10000

Class 0 (stem_high): 1 candidates
  Selected : (1, 1600)
  Sparsity : 72.1%
  Saved → /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/scVI/PDO/scvi_synthetic_stem_high_expression.csv

Class 1 (diff_low): 5382 candidates
  Selected : (1415, 1600)
  Sparsity : 98.9%
  Saved → /content/

scVI Generation Strategy — Posterior Sampling
--------------------------------------------
The original scVI paper (Lopez et al., 2018) was designed for normalization, batch correction, and differential expression — not for class-conditional synthetic data generation. Its generative process samples from an unconditional prior z ~ N(0,I), which produces a mixed pool with no class identity.
For this benchmark, we require class-specific synthetic cells (B cells and Monocytes separately) to enable fair comparison with class-conditional methods (SynCellNet, cscGAN). We achieve this using posterior sampling:

Real cells of each class are passed through the scVI encoder to obtain their posterior distributions q(z|x) in latent space
New latent vectors are sampled from those class-specific posteriors with added stochastic noise (reparameterisation trick)
These latent vectors are decoded through the trained scVI decoder to produce synthetic gene expression counts

This approach is fully consistent with the scVI paper — the encoder, decoder, and ZINB generative model are used exactly as trained, with no modification. The only difference from the paper's original generation is that we sample z from the posterior of real class-specific cells rather than the unconditional prior, which ensures class identity is preserved. This is a standard VAE-based data augmentation strategy.

In [4]:
# ============================================================
# scVI — Generate Synthetic Cells
# Google Colab | TensorFlow 2
# ============================================================

!pip install -q scikit-learn

# ── Cell 2: Imports ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print(f'TensorFlow: {tf.__version__}')

# ── Cell 3: Configuration — EDIT PATHS ───────────────────────
DATASET = 'PBMC'   # 'PBMC' or 'PDO'

REAL_DATA = {
    'PBMC': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/b_Class_dataset.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PBMC dataset/mono_Class_dataset.csv',
    },
    'PDO': {
        0: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Stem_High_Raw_Finalized.csv',
        1: '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Real PDO/3. Differential_Low_Raw_Finalized.csv',
    },
}

# Checkpoint dir — must contain best_model.index + best_model.data-00000-of-00001
CKPT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scVI Approach/PBMC/checkpoints',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scVI Approach/PDO/checkpoints',
}

OUTPUT_DIR = {
    'PBMC': '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scVI/PBMC',
    'PDO' : '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PDO dataset/scVI/PDO',
}

N_GENERATE  = 10000
MIN_PROB    = 0.90
N_PER_CLASS = {'PBMC': 1186, 'PDO': 1415}

CLASS_NAMES = {
    'PBMC': {0: 'b',         1: 'mono'},
    'PDO' : {0: 'stem_high', 1: 'diff_low'},
}

CFG  = dict(n_latent=10, n_hidden=128, n_layers=1, dropout_rate=0.1, dispersion='gene')
EPS  = 1e-8
SEED = 42

# ── Cell 4: scVI Architecture ─────────────────────────────────

class FCBlock(keras.layers.Layer):
    def __init__(self, n_out, dropout_rate=0.1, **kw):
        super().__init__(**kw)
        self.dense = keras.layers.Dense(n_out)
        self.bn    = keras.layers.BatchNormalization(momentum=0.99, epsilon=0.001)
        self.relu  = keras.layers.ReLU()
        self.drop  = keras.layers.Dropout(dropout_rate)
    def call(self, x, training=False):
        return self.drop(self.relu(self.bn(self.dense(x), training=training)),
                         training=training)

class scVI(keras.Model):
    def __init__(self, n_input, n_hidden=128, n_latent=10, n_layers=1,
                 dropout_rate=0.1, **kw):
        super().__init__(**kw)
        self.n_latent  = n_latent
        self.n_input   = n_input
        self.log_theta = tf.Variable(tf.random.normal([n_input]),
                                     trainable=True, name='log_theta')
        self.kl_weight = tf.Variable(0.0, trainable=False)

        self.enc_layers = [FCBlock(n_hidden, dropout_rate) for _ in range(n_layers)]
        self.l_mean_fc  = keras.layers.Dense(1)
        self.l_var_fc   = keras.layers.Dense(1, activation='softplus')
        self.z_mean_fc  = keras.layers.Dense(n_latent)
        self.z_var_fc   = keras.layers.Dense(n_latent, activation='softplus')

        self.dec_layers    = [FCBlock(n_hidden, dropout_rate) for _ in range(n_layers)]
        self.px_scale_fc   = keras.layers.Dense(n_input, activation='softmax')
        self.px_dropout_fc = keras.layers.Dense(n_input)

    def encode(self, x, training=False):
        h = tf.math.log1p(x)
        for layer in self.enc_layers:
            h = layer(h, training=training)
        return (self.z_mean_fc(h), self.z_var_fc(h) + EPS,
                self.l_mean_fc(h), self.l_var_fc(h) + EPS)

    def decode(self, z, log_l, training=False):
        h = z
        for layer in self.dec_layers:
            h = layer(h, training=training)
        px_rate = tf.exp(log_l) * self.px_scale_fc(h)
        return px_rate, self.px_dropout_fc(h)

    def reparameterise(self, mean, var):
        return mean + tf.sqrt(var) * tf.random.normal(tf.shape(mean))

    def compute_elbo(self, x, l_mean_prior, l_var_prior, training=False):
        z_mean, z_var, l_mean, l_var = self.encode(x, training=training)
        z     = self.reparameterise(z_mean, z_var)
        log_l = self.reparameterise(l_mean, l_var)
        px_rate, px_dropout = self.decode(z, log_l, training=training)
        theta = tf.exp(self.log_theta)
        recon = self._log_zinb(x, px_rate, theta, px_dropout)
        kl_z  = -0.5 * tf.reduce_sum(
            1 + tf.math.log(z_var) - tf.square(z_mean) - z_var, axis=1)
        kl_l  = -0.5 * tf.reduce_sum(
            1 + tf.math.log(z_var + EPS) - tf.math.log(l_var_prior + EPS)
            - (tf.square(l_mean - l_mean_prior) + z_var) / (l_var_prior + EPS), axis=1)
        elbo  = -tf.reduce_mean(recon) + self.kl_weight * tf.reduce_mean(kl_z + kl_l)
        return elbo, recon, kl_z, kl_l

    def _log_zinb(self, x, mu, theta, pi_logits):
        softplus_pi      = tf.nn.softplus(-pi_logits)
        log_theta_mu_eps = tf.math.log(theta + mu + EPS)
        nb_term = (theta * (tf.math.log(theta + EPS) - log_theta_mu_eps)
                   + x * (tf.math.log(mu + EPS) - log_theta_mu_eps)
                   + tf.math.lgamma(x + theta)
                   - tf.math.lgamma(theta)
                   - tf.math.lgamma(x + 1))
        log_p = tf.where(x < EPS,
                         tf.math.softplus(nb_term - pi_logits + softplus_pi) - softplus_pi,
                         nb_term - pi_logits + softplus_pi)
        return tf.reduce_sum(log_p, axis=1)

    def generate(self, n_cells, l_mean_prior, l_std_prior):
        z     = tf.random.normal([n_cells, self.n_latent])
        log_l = tf.random.normal([n_cells, 1]) * l_std_prior + l_mean_prior
        px_rate, _ = self.decode(z, log_l, training=False)
        theta = tf.exp(self.log_theta)
        concentration = tf.broadcast_to(theta, tf.shape(px_rate))
        rate          = theta / (px_rate + EPS)
        gamma_samples = tf.random.gamma(shape=[], alpha=concentration, beta=rate)
        counts        = tf.random.poisson(shape=[], lam=gamma_samples + EPS)
        return counts.numpy().astype(np.float32)

# ── Cell 5: Load Real Data ────────────────────────────────────

print(f'Loading real {DATASET} data...')
Xs, ys = [], []
for label_idx, path in REAL_DATA[DATASET].items():
    df = pd.read_csv(path, header=None)
    Xs.append(df.values.astype(np.float32))
    ys.append(np.full(len(df), label_idx, dtype=np.int32))
    print(f'  Class {label_idx}: {df.shape}')

X_real  = np.concatenate(Xs)
y_real  = np.concatenate(ys)
N_GENES = X_real.shape[1]
log_lib  = np.log(X_real.sum(axis=1) + EPS)
lib_mean = float(log_lib.mean())
lib_std  = float(log_lib.std())
print(f'Combined: {X_real.shape} | lib_mean={lib_mean:.3f}, lib_std={lib_std:.3f}')

# ── Cell 6: Build scVI & Restore Checkpoint ──────────────────

tf.random.set_seed(SEED)
model = scVI(N_GENES, CFG['n_hidden'], CFG['n_latent'],
             CFG['n_layers'], CFG['dropout_rate'], name=f'scVI_{DATASET}')

# Build weights via dummy forward pass
_ = model.compute_elbo(tf.zeros([2, N_GENES]), tf.zeros([2,1]), tf.ones([2,1]))
print('scVI model ready.')

ckpt   = tf.train.Checkpoint(model=model)
status = ckpt.restore(os.path.join(CKPT_DIR[DATASET], 'best_model')).expect_partial()
print('Checkpoint restored.')

# ── Cell 7: Generate Via Posterior Sampling Per Class ────────
# Instead of sampling from prior N(0,I), we sample from the
# posterior q(z|x) of real cells per class.
# This keeps generation class-specific without needing a classifier.

def generate_from_posterior(model, X_class, n_cells, lib_mean, lib_std, seed=42):
    """
    Encode real cells to get posterior q(z|x),
    sample new z from those posteriors,
    decode to get synthetic counts.
    """
    tf.random.set_seed(seed)
    np.random.seed(seed)

    X_tf = tf.constant(X_class, dtype=tf.float32)

    # Encode all real cells of this class
    z_mean, z_var, l_mean, l_var = model.encode(X_tf, training=False)
    z_mean = z_mean.numpy()
    z_var  = z_var.numpy()

    n_real = len(X_class)

    # Sample n_cells cells with replacement from real cells,
    # then sample z from their posteriors
    rng     = np.random.default_rng(seed)
    indices = rng.choice(n_real, size=n_cells, replace=(n_cells > n_real))

    sampled_mean = z_mean[indices]                          # (n_cells, n_latent)
    sampled_std  = np.sqrt(z_var[indices])                  # (n_cells, n_latent)
    eps          = rng.standard_normal(sampled_mean.shape)
    z_samples    = sampled_mean + sampled_std * eps         # reparameterisation

    # Sample library size from real library distribution
    log_l_samples = rng.normal(lib_mean, lib_std, size=(n_cells, 1)).astype(np.float32)

    # Decode
    z_tf   = tf.constant(z_samples, dtype=tf.float32)
    log_l  = tf.constant(log_l_samples, dtype=tf.float32)
    px_rate, _ = model.decode(z_tf, log_l, training=False)

    # Sample counts from NB (Gamma-Poisson mixture)
    theta         = tf.exp(model.log_theta).numpy()
    px_rate_np    = px_rate.numpy()
    concentration = np.broadcast_to(theta, px_rate_np.shape)
    rate          = theta / (px_rate_np + EPS)

    rng2          = np.random.default_rng(seed + 1)
    gamma_samples = rng2.gamma(shape=concentration, scale=1.0/rate)
    counts        = rng2.poisson(lam=gamma_samples + EPS).astype(np.float32)

    return counts


os.makedirs(OUTPUT_DIR[DATASET], exist_ok=True)
n_per_class = N_PER_CLASS[DATASET]

for class_idx, class_name in CLASS_NAMES[DATASET].items():
    print(f'\nGenerating {n_per_class} cells — class {class_idx} ({class_name})...')

    # Get real cells for this class
    X_class = Xs[class_idx]

    synthetic = generate_from_posterior(
        model, X_class, n_per_class, lib_mean, lib_std, seed=SEED + class_idx)

    print(f'  Shape   : {synthetic.shape}')
    print(f'  Range   : [{synthetic.min():.1f}, {synthetic.max():.1f}]')
    print(f'  Sparsity: {(synthetic == 0).mean()*100:.1f}%')

    out_path = os.path.join(OUTPUT_DIR[DATASET],
                            f'scvi_synthetic_{class_name}_expression.csv')
    pd.DataFrame(synthetic).to_csv(out_path, header=False, index=False)
    print(f'  Saved → {out_path}')

print('\nDone.')

TensorFlow: 2.20.0
Loading real PBMC data...
  Class 0: (1186, 1600)
  Class 1: (1186, 1600)
Combined: (2372, 1600) | lib_mean=5.947, lib_std=0.454
scVI model ready.
Checkpoint restored.

Generating 1186 cells — class 0 (b)...
  Shape   : (1186, 1600)
  Range   : [0.0, 43.0]
  Sparsity: 80.3%
  Saved → /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scVI/PBMC/scvi_synthetic_b_expression.csv

Generating 1186 cells — class 1 (mono)...
  Shape   : (1186, 1600)
  Range   : [0.0, 51.0]
  Sparsity: 80.4%
  Saved → /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scVI/PBMC/scvi_synthetic_mono_expression.csv

Done.
